In [ ]:
from __future__ import annotations

import os
import sys
import pickle
import logging
import warnings
import subprocess
import importlib
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

# ─────────────────────────────────────────────────────────────────────────────
# 1.  DEPENDENCY BOOTSTRAP
# ─────────────────────────────────────────────────────────────────────────────
_REQUIRED: Dict[str, str] = {
    "pandas":   "pandas>=2.0",
    "numpy":    "numpy>=1.24",
    "catboost": "catboost>=1.2",
    "sklearn":  "scikit-learn>=1.3",
    "rich":     "rich>=13.0",
    "requests": "requests>=2.31",
}


def _bootstrap() -> None:
    """Install any missing packages silently before the main imports."""
    missing = [pip for imp, pip in _REQUIRED.items()
               if importlib.util.find_spec(imp) is None]
    if missing:
        print(f"📦 Installing: {', '.join(missing)} …")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q"] + missing,
            stdout=subprocess.DEVNULL,
        )


_bootstrap()

import numpy as np
import pandas as pd
import requests
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import (
    accuracy_score, f1_score,
    mean_absolute_error, mean_squared_error, r2_score,
)
from sklearn.model_selection import train_test_split
from rich import box
from rich.console import Console
from rich.panel import Panel
from rich.progress import (
    BarColumn, Progress, SpinnerColumn,
    TextColumn, TimeElapsedColumn,
)
from rich.prompt import Prompt
from rich.table import Table
from rich.theme import Theme

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# 2.  THEME  &  CONSOLE
# ─────────────────────────────────────────────────────────────────────────────
_THEME = Theme({
    "header":  "bold bright_blue",
    "section": "bold cyan",
    "ok":      "bold green",
    "warn":    "bold yellow",
    "error":   "bold red",
    "dim":     "dim white",
    "price":   "bold bright_green",
    "ontime":  "bold green",
    "delayed": "bold red",
    "reason":  "bold yellow",
})

console = Console(theme=_THEME)
log = logging.getLogger("railway")

# ─────────────────────────────────────────────────────────────────────────────
# 3.  CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
class Config:
    """All static constants in one place — easy to modify without touching logic."""

    # ── File paths ────────────────────────────────────────────────────────────
    CSV_URL: str = (
        "https://raw.githubusercontent.com/mszakii/DEPI-CAPSTONE/"
        "refs/heads/master/cleaned_railway.csv"
    )
    CSV_PATH:     Path = Path.cwd() / "cleaned_railway.csv"
    MODEL_CACHE:  Path = Path.cwd() / "railway_models_v4.pkl"
    CACHE_VERSION: str = "4.0.0"   # bump this to force a retrain
    RANDOM_STATE:  int = 42

    # ── Feature columns ───────────────────────────────────────────────────────
    CAT_COLS: Tuple[str] = (
        "Departure Station", "Arrival Destination", "Route",
        "Railcard", "Ticket Type", "Ticket Class",
    )
    NUM_COLS: Tuple[str, ...] = (
        "Lead Time",
        "Journey_Month", "Journey_DayOfWeek",
        "Journey_DayOfMonth", "Journey_IsWeekend",
        "Journey_IsPeakHour", "Journey_Season",
        "Departure_Hour",
    )

    @classmethod
    def all_features(cls) -> List[str]:
        return list(cls.CAT_COLS) + list(cls.NUM_COLS)

    @classmethod
    def cat_indices(cls) -> List[int]:
        feats = cls.all_features()
        return [feats.index(c) for c in cls.CAT_COLS]

    # ── Model hyperparameters ─────────────────────────────────────────────────
    PRICE_PARAMS: Dict[str, Any] = dict(
        iterations=1200, learning_rate=0.04, depth=6,
        l2_leaf_reg=3, min_data_in_leaf=20,
    )
    STATUS_PARAMS: Dict[str, Any] = dict(
        iterations=1500, learning_rate=0.03, depth=8,
        auto_class_weights="Balanced", l2_leaf_reg=5,
    )
    DELAY_PARAMS: Dict[str, Any] = dict(
        iterations=1500, learning_rate=0.03, depth=8,
        auto_class_weights="Balanced", l2_leaf_reg=5,
    )
    EARLY_STOP: int = 100

    # ── Domain values (used in menus) ─────────────────────────────────────────
    DEPARTURE_STATIONS: Tuple[str] = (
        "Birmingham New Street", "Bristol Temple Meads",
        "Edinburgh Waverley",    "Liverpool Lime Street",
        "London Euston",         "London Kings Cross",
        "London Paddington",     "London St Pancras",
        "Manchester Piccadilly", "Oxford",
        "Reading",               "York",
    )
    ARRIVAL_STATIONS: Tuple[str] = (
        "Birmingham New Street", "Bristol Temple Meads",  "Cardiff Central",
        "Coventry",              "Crewe",                 "Didcot",
        "Doncaster",             "Durham",                "Edinburgh Waverley",
        "Leeds",                 "Leicester",             "Liverpool Lime Street",
        "London Euston",         "London Kings Cross",    "London Paddington",
        "London St Pancras",     "London Waterloo",       "Manchester Piccadilly",
        "Nottingham",            "Nuneaton",              "Oxford",
        "Peterborough",          "Reading",               "Sheffield",
        "Stafford",              "Swindon",               "Tamworth",
        "Wakefield",             "Warrington",            "Wolverhampton",
        "York",
    )
    TICKET_CLASSES: Tuple[str] = ("Standard", "First Class")
    TICKET_TYPES:   Tuple[str] = ("Advance", "Off-Peak", "Anytime")
    RAILCARDS:      Tuple[str] = ("No Railcard", "Adult", "Disabled", "Senior")

    # Peak hours: 07-09 and 17-19
    PEAK_HOURS: frozenset = frozenset(range(7, 10)) | frozenset(range(17, 20))


# ─────────────────────────────────────────────────────────────────────────────
# 4.  DATA CLASSES
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class JourneyInput:
    """Validated user inputs for a single journey."""

    departure:    str
    arrival:      str
    journey_date: datetime
    hour:         int
    lead_time:    int
    ticket_class: str
    ticket_type:  str
    railcard:     str

    # ── Derived helpers ───────────────────────────────────────────────────────
    @property
    def route(self) -> str:
        return f"{self.departure} to {self.arrival}"

    @property
    def is_peak(self) -> bool:
        return self.hour in Config.PEAK_HOURS

    @property
    def season(self) -> int:
        """0 = winter · 1 = spring · 2 = summer · 3 = autumn"""
        return (self.journey_date.month % 12) // 3

    def to_feature_dict(self) -> Dict[str, Any]:
        d = self.journey_date
        return {
            "Departure Station":   self.departure,
            "Arrival Destination": self.arrival,
            "Route":               self.route,
            "Railcard":            self.railcard,
            "Ticket Type":         self.ticket_type,
            "Ticket Class":        self.ticket_class,
            "Lead Time":           self.lead_time,
            "Journey_Month":       d.month,
            "Journey_DayOfWeek":   d.weekday(),
            "Journey_DayOfMonth":  d.day,
            "Journey_IsWeekend":   int(d.weekday() >= 5),
            "Journey_IsPeakHour":  int(self.is_peak),
            "Journey_Season":      self.season,
            "Departure_Hour":      self.hour,
        }


@dataclass
class PredictionResult:
    """All model outputs for one journey."""

    price:       float
    status:      str    # "On Time" | "Disrupted"
    status_conf: float  # 0 – 100
    reason:      str    # delay reason label or "N/A"
    reason_conf: float  # 0 – 100
    is_unseen:   bool
    journey:     JourneyInput


@dataclass
class JourneyAssessment:
    journey_score:  int
    risk_score:     int
    value_score:    int
    recommendation: str
    rating:         str
    booking_advice: str


# ─────────────────────────────────────────────────────────────────────────────
# 5.  FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive temporal and route features from raw CSV columns.
    New in v4: Journey_IsPeakHour, Journey_Season.
    """
    X  = df.copy()
    dt = pd.to_datetime(X["Date of Journey"], errors="coerce")

    X["Journey_Month"]     = dt.dt.month
    X["Journey_DayOfWeek"] = dt.dt.dayofweek
    X["Journey_DayOfMonth"]= dt.dt.day
    X["Journey_IsWeekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)

    dep_hour = (
        pd.to_datetime(X["Departure Time"], format="%H:%M:%S", errors="coerce")
        .dt.hour.fillna(8).astype(int)
    )
    X["Departure_Hour"]     = dep_hour
    X["Journey_IsPeakHour"] = dep_hour.isin(Config.PEAK_HOURS).astype(int)
    X["Journey_Season"]     = (X["Journey_Month" ] % 12 // 3)
    X["Route"]              = X["Departure Station"] + " to " + X["Arrival Destination"]

    return X


# ─────────────────────────────────────────────────────────────────────────────
# 6.  MODEL BUNDLE
# ─────────────────────────────────────────────────────────────────────────────
class ModelBundle:
    """
    Encapsulates the three trained CatBoost models + metadata.
    Exposes a single `predict(journey) → PredictionResult` method.
    """
    def assess_journey(self, result: PredictionResult) -> JourneyAssessment:

        score = 100

        # Price impact
        if result.price > 120:
            score -= 30
        elif result.price > 80:
            score -= 20
        elif result.price > 50:
            score -= 10

        # Delay impact
        if result.status == "Disrupted":
            score -= min(40, result.status_conf * 0.4)

        # Peak impact
        if result.journey.is_peak:
            score -= 10

        # Lead time impact
        if result.journey.lead_time < 3:
            score -= 15
        elif result.journey.lead_time < 7:
            score -= 8

        journey_score = max(0, min(100, score))

        risk_score = 100 - journey_score

        value_score = max(
            0,
            min(
                100,
                100 - (result.price * 0.8)
            )
        )

        if journey_score >= 85:
            rating = "Excellent"
            recommendation = (
                "Excellent journey. "
                "Low disruption risk and good overall value."
            )

        elif journey_score >= 70:
            rating = "Good"
            recommendation = (
                "Good journey with acceptable risk."
            )

        elif journey_score >= 50:
            rating = "Average"
            recommendation = (
                "Moderate risk. Consider alternatives."
            )

        else:
            rating = "Poor"
            recommendation = (
                "High risk or poor value."
            )

        if journey_score >= 80:
            booking_advice = "BOOK NOW"
        elif journey_score >= 60:
            booking_advice = "CONSIDER"
        else:
            booking_advice = "LOOK FOR ALTERNATIVES"

        return JourneyAssessment(
            journey_score=journey_score,
            risk_score=risk_score,
            value_score=value_score,
            recommendation=recommendation,
            rating=rating,
            booking_advice=booking_advice,
        )

    def __init__(
        self,
        price_model:  CatBoostRegressor,
        status_model: CatBoostClassifier,
        delay_model:  CatBoostClassifier,
        valid_routes: set,
        metrics:      Dict[str, float],
        version:      str,
        trained_at:   datetime,
    ) -> None:
        self.price_model  = price_model
        self.status_model = status_model
        self.delay_model  = delay_model
        self.valid_routes = valid_routes
        self.metrics      = metrics
        self.version      = version
        self.trained_at   = trained_at

    # ── Inference ─────────────────────────────────────────────────────────────
    def predict(self, journey: JourneyInput) -> PredictionResult:
        X = pd.DataFrame([journey.to_feature_dict()])[Config.all_features()]
        is_unseen = (journey.departure, journey.arrival) not in self.valid_routes

        # Price
        price = max(1.0, round(float(self.price_model.predict(X)[0]), 2))

        # Journey status
        s_proba = self.status_model.predict_proba(X)[0]
        s_idx   = int(self.status_model.predict(X)[0])
        status  = "Disrupted" if s_idx == 1 else "On Time"
        s_conf  = float(s_proba[s_idx]) * 100

        # Delay reason (only when disrupted)
        reason, d_conf = "N/A", 0.0
        if status == "Disrupted":
            d_raw   = self.delay_model.predict(X)[0]
            d_proba = self.delay_model.predict_proba(X)[0]
            d_idx   = self.delay_model.classes_.tolist().index(d_raw)
            reason  = str(d_raw)
            d_conf  = float(d_proba[d_idx]) * 100

        return PredictionResult(
            price=price, status=status, status_conf=s_conf,
            reason=reason, reason_conf=d_conf,
            is_unseen=is_unseen, journey=journey,
        )


# ─────────────────────────────────────────────────────────────────────────────
# 7.  TRAINING
# ─────────────────────────────────────────────────────────────────────────────
def _fit_price_model(
    X_tr: pd.DataFrame, y_tr: pd.Series,
    X_te: pd.DataFrame, y_te: pd.Series,
    cat_idx: List[int],
    progress: Progress,
) -> Tuple[CatBoostRegressor, Dict[str, float]]:

    task = progress.add_task("[cyan]Training Price model (1/3)…", total=1)
    model = CatBoostRegressor(
        **Config.PRICE_PARAMS,
        random_seed=Config.RANDOM_STATE,
        verbose=0,
    )
    model.fit(X_tr, y_tr, cat_features=cat_idx)
    preds = model.predict(X_te)
    metrics = {
        "price_mae":  mean_absolute_error(y_te, preds),
        "price_rmse": float(mean_squared_error(y_te, preds) ** 0.5),
        "price_r2":   r2_score(y_te, preds),
    }
    progress.update(task, completed=1)
    return model, metrics


def _fit_status_model(
    X_tr: pd.DataFrame, y_tr: pd.Series,
    X_te: pd.DataFrame, y_te: pd.Series,
    cat_idx: List[int],
    progress: Progress,
) -> Tuple[CatBoostClassifier, Dict[str, float]]:

    task = progress.add_task("[cyan]Training Journey Status model (2/3)…", total=1)
    model = CatBoostClassifier(
        **Config.STATUS_PARAMS,
        random_seed=Config.RANDOM_STATE,
        verbose=0,
    )
    model.fit(
        X_tr, y_tr, cat_features=cat_idx,
        eval_set=(X_te, y_te),
        early_stopping_rounds=Config.EARLY_STOP,
    )
    preds = model.predict(X_te)
    metrics = {
        "status_acc": accuracy_score(y_te, preds),
        "status_f1":  f1_score(y_te, preds, average="weighted"),
    }
    progress.update(task, completed=1)
    return model, metrics


def _fit_delay_model(
    df_eng: pd.DataFrame,
    cat_idx: List[int],
    progress: Progress,
) -> Tuple[CatBoostClassifier, Dict[str, float]]:

    task = progress.add_task("[cyan]Training Delay Reason model (3/3)…", total=1)
    df_delay = df_eng[df_eng["Journey Status"] != "On Time"]
    X_d, y_d = df_delay[Config.all_features()], df_delay["Reason for Delay"]
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_d, y_d, test_size=0.2,
        stratify=y_d, random_state=Config.RANDOM_STATE,
    )
    model = CatBoostClassifier(
        **Config.DELAY_PARAMS,
        random_seed=Config.RANDOM_STATE,
        verbose=0,
    )
    model.fit(
        X_tr, y_tr, cat_features=cat_idx,
        eval_set=(X_te, y_te),
        early_stopping_rounds=Config.EARLY_STOP,
    )
    preds = model.predict(X_te)
    metrics = {
        "delay_acc": accuracy_score(y_te, preds),
        "delay_f1":  f1_score(y_te, preds, average="weighted"),
    }
    progress.update(task, completed=1)
    return model, metrics


def train_system(df: pd.DataFrame) -> ModelBundle:
    """
    Full training pipeline:
      1. Feature engineering
      2. Chronological 80/20 train-test split
      3. Fit Price, Status, Delay models
      4. Return a ModelBundle with metrics
    """
    df_eng    = engineer_features(df)
    df_sorted = df_eng.sort_values("Date of Journey").reset_index(drop=True)
    split     = int(len(df_sorted) * 0.8)

    all_feat  = Config.all_features()
    cat_idx   = Config.cat_indices()
    X_all     = df_sorted[all_feat]
    X_tr, X_te = X_all.iloc[:split], X_all.iloc[split:]

    y_price  = df_sorted["Price"]
    y_status = (df_sorted["Journey Status"] != "On Time").astype(int)

    with Progress(
        SpinnerColumn(),
        TextColumn("[progress.description]{task.description}"),
        BarColumn(bar_width=28),
        TimeElapsedColumn(),
        console=console,
    ) as progress:
        m_price,  m_price_met  = _fit_price_model(
            X_tr, y_price.iloc[:split],
            X_te, y_price.iloc[split:],
            cat_idx, progress,
        )
        m_status, m_status_met = _fit_status_model(
            X_tr, y_status.iloc[:split],
            X_te, y_status.iloc[split:],
            cat_idx, progress,
        )
        m_delay,  m_delay_met  = _fit_delay_model(df_eng, cat_idx, progress)

    valid_routes = set(zip(df_sorted["Departure Station"], df_sorted["Arrival Destination"]))
    metrics      = {**m_price_met, **m_status_met, **m_delay_met}

    return ModelBundle(
        price_model=m_price,
        status_model=m_status,
        delay_model=m_delay,
        valid_routes=valid_routes,
        metrics=metrics,
        version=Config.CACHE_VERSION,
        trained_at=datetime.now(),
    )


# ─────────────────────────────────────────────────────────────────────────────
# 8.  CACHE  (load / train / save)
# ─────────────────────────────────────────────────────────────────────────────
def _download_csv() -> None:
    """Download the CSV from GitHub if not already present."""
    console.print(f"[dim]↓  Downloading dataset from GitHub…[/dim]")
    try:
        resp = requests.get(Config.CSV_URL, timeout=60)
        resp.raise_for_status()
        Config.CSV_PATH.write_bytes(resp.content)
        console.print("[ok]✓ Dataset downloaded successfully.[/ok]")
    except requests.RequestException as exc:
        console.print(f"[error]✗ Download failed: {exc}[/error]")
        sys.exit(1)


def get_bundle() -> ModelBundle:
    """
    Return a ModelBundle from cache if version matches,
    otherwise retrain from scratch and persist.
    """
    # ── Try cache ─────────────────────────────────────────────────────────────
    if Config.MODEL_CACHE.exists():
        try:
            with open(Config.MODEL_CACHE, "rb") as fh:
                bundle: ModelBundle = pickle.load(fh)
            if getattr(bundle, "version", None) == Config.CACHE_VERSION:
                console.print(
                    f"[dim]✓ Loaded cached models "
                    f"(v{bundle.version}, "
                    f"trained {bundle.trained_at.strftime('%Y-%m-%d %H:%M')}).[/dim]"
                )
                return bundle
            console.print("[warn]⚠  Cache version mismatch — retraining…[/warn]")
        except Exception as exc:
            console.print(f"[warn]⚠  Cache unreadable ({exc}) — retraining…[/warn]")

    # ── Fresh training ────────────────────────────────────────────────────────
    console.print("[warn]No valid cache — training models (≈ 1–3 min)…[/warn]")
    if not Config.CSV_PATH.exists():
        _download_csv()

    df = pd.read_csv(Config.CSV_PATH)
    console.print(f"[dim]Dataset: {len(df):,} rows × {len(df.columns)} columns[/dim]")

    bundle = train_system(df)

    with open(Config.MODEL_CACHE, "wb") as fh:
        pickle.dump(bundle, fh)
    console.print(f"[ok]✓ Models cached → {Config.MODEL_CACHE}[/ok]")
    return bundle


# ─────────────────────────────────────────────────────────────────────────────
# 9.  UI  HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def conf_bar(pct: float, width: int = 24) -> str:
    """Return a coloured block-bar string representing a confidence %."""
    filled = int(round(pct / 100 * width))
    bar    = "█" * filled + "░" * (width - filled)
    color  = "green" if pct >= 70 else "yellow" if pct >= 50 else "red"
    return f"[{color}]{bar}[/{color}] {pct:.1f}%"


def _rating(value: float, good: float, ok: float, inverted: bool = False) -> str:
    """Return a coloured 'Excellent / Good / Fair' badge."""
    if not inverted:
        tier = "Excellent" if value >= good else ("Good" if value >= ok else "Fair")
        color = "green"    if value >= good else ("yellow" if value >= ok else "red")
    else:
        # Lower is better (e.g. MAE, RMSE)
        tier = "Excellent" if value <= good else ("Good" if value <= ok else "Fair")
        color = "green"    if value <= good else ("yellow" if value <= ok else "red")
    return f"[{color}]{tier}[/{color}]"


def select_menu(
    title:   str,
    options: Tuple[str, ...] | List[str],
    default: int = 0,
) -> str:
    """Numbered selection menu with a highlighted default."""
    console.print(f"\n[section]{title}[/section]")
    for i, opt in enumerate(options, 1):
        marker = "[cyan]►[/cyan]" if i == default + 1 else " "
        console.print(f"  {marker} [dim]{i:2d}.[/dim] {opt}")
    while True:
        raw = Prompt.ask(f"  Select [1-{len(options)}]", default=str(default + 1))
        if raw.isdigit() and 1 <= int(raw) <= len(options):
            return options[int(raw) - 1]
        console.print(f"  [error]✗ Enter a number between 1 and {len(options)}.[/error]")


def ask_int(prompt: str, lo: int, hi: int, default: int) -> int:
    """Prompt for an integer in [lo, hi]; loops on bad input."""
    while True:
        raw = Prompt.ask(prompt, default=str(default))
        if raw.isdigit() and lo <= int(raw) <= hi:
            return int(raw)
        console.print(f"  [error]✗ Enter a whole number between {lo} and {hi}.[/error]")


def ask_date(prompt: str) -> datetime:
    """Prompt for a YYYY-MM-DD date; loops on bad format."""
    while True:
        raw = Prompt.ask(prompt, default=datetime.now().strftime("%Y-%m-%d"))
        try:
            return datetime.strptime(raw.strip(), "%Y-%m-%d")
        except ValueError:
            console.print("  [error]✗ Use YYYY-MM-DD format (e.g. 2025-06-15).[/error]")


def print_metrics(m: Dict[str, float]) -> None:
    """Render a colour-rated metrics table."""
    t = Table(box=box.SIMPLE_HEAD, show_header=True, header_style="bold dim", padding=(0, 2))
    t.add_column("Model",   style="dim", min_width=18)
    t.add_column("Metric",  style="dim", min_width=6)
    t.add_column("Value",   justify="right", min_width=10)
    t.add_column("Rating",  min_width=12)

    t.add_row("Price",          "MAE",  f"£{m['price_mae']:.2f}",     _rating(m["price_mae"],  5.0,  15.0, inverted=True))
    t.add_row("Price",          "RMSE", f"£{m['price_rmse']:.2f}",    _rating(m["price_rmse"], 8.0,  20.0, inverted=True))
    t.add_row("Price",          "R²",   f"{m['price_r2']:.3f}",       _rating(m["price_r2"],  0.85, 0.70))
    t.add_row("Journey Status", "Acc",  f"{m['status_acc']*100:.1f}%%",_rating(m["status_acc"],0.85, 0.70))
    t.add_row("Journey Status", "F1",   f"{m['status_f1']:.3f}",      _rating(m["status_f1"], 0.85, 0.70))
    t.add_row("Delay Reason",   "Acc",  f"{m['delay_acc']*100:.1f}%%", _rating(m["delay_acc"], 0.75, 0.60))
    t.add_row("Delay Reason",   "F1",   f"{m['delay_f1']:.3f}",       _rating(m["delay_f1"],  0.75, 0.60))
    console.print(t)


# ─────────────────────────────────────────────────────────────────────────────
# 10. INPUT COLLECTION
# ─────────────────────────────────────────────────────────────────────────────
def get_journey_input() -> JourneyInput:
    """
    Gather all journey details from the user interactively.
    Loops until departure ≠ arrival.
    """
    while True:
        dep = select_menu("Departure Station",   Config.DEPARTURE_STATIONS, 4)
        arr = select_menu("Arrival Destination", Config.ARRIVAL_STATIONS,  17)
        if dep != arr:
            break
        console.print("  [error]✗ Departure and arrival must be different — try again.[/error]")

    jdate  = ask_date(
        "\n[section]Date of Journey[/section] [dim](YYYY-MM-DD)[/dim]\n  Date"
    )
    hour   = ask_int(
        "\n[section]Departure Hour[/section] [dim](0 = midnight · 12 = noon · 23 = 11 pm)[/dim]\n  Hour [0–23]",
        0, 23, 8,
    )
    lead   = ask_int(
        "\n[section]Lead Time[/section] [dim](days booked in advance)[/dim]\n  Days [0–365]",
        0, 365, 7,
    )
    tclass = select_menu("Ticket Class", Config.TICKET_CLASSES)
    ttype  = select_menu("Ticket Type",  Config.TICKET_TYPES)
    rcard  = select_menu("Railcard",     Config.RAILCARDS)

    return JourneyInput(
        departure=dep, arrival=arr,
        journey_date=jdate, hour=hour, lead_time=lead,
        ticket_class=tclass, ticket_type=ttype, railcard=rcard,
    )


# ─────────────────────────────────────────────────────────────────────────────
# 11. RESULT DISPLAY
# ─────────────────────────────────────────────────────────────────────────────
def display_result(result: PredictionResult, n: int, bundle: ModelBundle) -> None:
    """Render the full prediction output for one journey."""
    j = result.journey
    console.print()
    console.rule(f"[header]  Prediction #{n}  [/header]")

    # ── Journey summary table ─────────────────────────────────────────────────
    info = Table(box=box.SIMPLE, show_header=False, padding=(0, 2))
    info.add_column("k", style="dim", min_width=18)
    info.add_column("v", style="bold")

    season_label = ["Winter ❄️", "Spring 🌱", "Summer ☀️", "Autumn 🍂"][j.season]
    peak_label   = "🌅 Peak hours" if j.is_peak else "🌙 Off-peak"

    info.add_row("Route",    f"{j.departure}  →  {j.arrival}")
    info.add_row("Date",     f"{j.journey_date.strftime('%d %b %Y')}  ({j.journey_date.strftime('%A')})  {season_label}")
    info.add_row("Departs",  f"{j.hour:02d}:00  {peak_label}")
    info.add_row("Lead Time",f"{j.lead_time} day{'s' if j.lead_time != 1 else ''} in advance")
    info.add_row("Class",    j.ticket_class)
    info.add_row("Type",     j.ticket_type)
    info.add_row("Railcard", j.railcard)
    if result.is_unseen:
        info.add_row(
            "⚠  Note",
            "[warn]Unseen route — extrapolation, treat with caution[/warn]",
        )
    console.print(info)
    assessment = bundle.assess_journey(result)

    score_table = Table(
        title="Journey Intelligence Report",
        box=box.ROUNDED
    )

    score_table.add_column("Metric")
    score_table.add_column("Value")

    score_table.add_row(
        "Journey Score",
        f"{assessment.journey_score:.0f}/100"
    )

    score_table.add_row(
        "Risk Score",
        f"{assessment.risk_score:.0f}/100"
    )

    score_table.add_row(
        "Value Score",
        f"{assessment.value_score:.0f}/100"
    )

    score_table.add_row(
        "Rating",
        assessment.rating
    )

    score_table.add_row(
        "Booking Advice",
        assessment.booking_advice
    )

    console.print(score_table)

    console.print(
        Panel(
            assessment.recommendation,
            title="AI Recommendation"
        )
    )

    # ── Predictions table ─────────────────────────────────────────────────────
    res = Table(
        box=box.ROUNDED, expand=True, show_header=True,
        header_style="bold white on dark_blue", padding=(0, 1),
    )
    res.add_column("Prediction", style="bold", min_width=22)
    res.add_column("Result",                   min_width=22)
    res.add_column("Confidence",               min_width=34)

    res.add_row(
        "💷  Estimated Price",
        f"[price]£{result.price:.2f}[/price]",
        f"[dim]{j.ticket_type} · {j.ticket_class}[/dim]",
    )

    s_style = "ontime" if result.status == "On Time" else "delayed"
    s_icon  = "✅" if result.status == "On Time" else "⚠️ "
    res.add_row(
        "🚦  Journey Status",
        f"[{s_style}]{s_icon} {result.status}[/{s_style}]",
        conf_bar(result.status_conf),
    )

    if result.status == "Disrupted":
        res.add_row(
            "🔍  Delay Reason",
            f"[reason]{result.reason}[/reason]",
            conf_bar(result.reason_conf),
        )

    console.print(res)

    # ── Plain-English verdict ─────────────────────────────────────────────────
    if result.status == "On Time":
        color   = "green"
        verdict = (
            f"Your [price]£{result.price:.2f}[/price] {j.ticket_type.lower()} "
            f"{j.ticket_class.lower()} ticket looks good — "
            f"[ontime]on-time journey expected[/ontime] "
            f"({result.status_conf:.0f}%% confidence)."
        )
    else:
        color   = "red"
        verdict = (
            f"Your [price]£{result.price:.2f}[/price] {j.ticket_type.lower()} "
            f"{j.ticket_class.lower()} ticket carries a disruption risk: "
            f"[delayed]{result.status}[/delayed] predicted due to "
            f"[reason]{result.reason}[/reason] "
            f"({result.reason_conf:.0f}%% confidence)."
        )
    console.print(Panel(verdict, border_style=color, padding=(0, 2)))


# ─────────────────────────────────────────────────────────────────────────────
# 12. SESSION SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
def display_session_summary(results: List[PredictionResult]) -> None:
    """Print a compact table of all predictions made in this session."""
    if not results:
        return

    console.print()
    console.rule("[header]  Session Summary  [/header]")

    t = Table(box=box.SIMPLE_HEAD, show_header=True, header_style="bold dim", padding=(0, 1))
    t.add_column("#",       style="dim",   justify="right", min_width=3)
    t.add_column("Route",                  min_width=36)
    t.add_column("Date",                   min_width=13)
    t.add_column("Price",  justify="right",min_width=8)
    t.add_column("Status", justify="center")
    t.add_column("Reason")

    for i, r in enumerate(results, 1):
        j      = r.journey
        status = "[ontime]On Time ✅[/ontime]" if r.status == "On Time" else "[delayed]Disrupted ⚠️[/delayed]"
        t.add_row(
            str(i),
            f"{j.departure[:18]} → {j.arrival[:18]}",
            j.journey_date.strftime("%d %b %Y"),
            f"£{r.price:.2f}",
            status,
            r.reason if r.status == "Disrupted" else "—",
        )
    console.print(t)

    prices  = [r.price for r in results]
    n_ok    = sum(1 for r in results if r.status == "On Time")
    n_bad   = len(results) - n_ok
    avg_p   = sum(prices) / len(prices)

    console.print(
        f"\n  Journeys: {len(results)}  |  "
        f"On-time: [ok]{n_ok}[/ok]  |  "
        f"Disrupted: [error]{n_bad}[/error]  |  "
        f"Avg price: [price]£{avg_p:.2f}[/price]  |  "
        f"Range: £{min(prices):.2f} – £{max(prices):.2f}"
    )


# ─────────────────────────────────────────────────────────────────────────────
# 13. MAIN LOOP
# ─────────────────────────────────────────────────────────────────────────────
def main() -> None:
    console.rule("[header]  682  Railway Journey Predictor v4  682  [/header]")
    console.print("[dim]  Powered by CatBoost · UK National Rail Data[/dim]\n")

    # Load or train models
    bundle = get_bundle()

    # Show model metrics
    if bundle.metrics:
        console.print("\n[bold]Model performance on held-out test data:[/bold]")
        print_metrics(bundle.metrics)

    session_results: List[PredictionResult] = []

    # ── Prediction loop ───────────────────────────────────────────────────────
    while True:
        console.rule("[section]  New Journey[/section]")
        journey = get_journey_input()

        # Run inference
        with console.status("[bold green]  Running models…[/bold green]", spinner="dots12"):
            result = bundle.predict(journey)

        session_results.append(result)
        display_result(result, len(session_results), bundle)

        # Post-prediction menu
        console.print(
            "\n  [dim](p)[/dim] Predict another  "
            "[dim](s)[/dim] Session summary  "
            "[dim](q)[/dim] Quit"
        )
        choice = Prompt.ask("  What next?", choices=["p", "s", "q"], default="p", show_choices=False)

        if choice == "s":
            display_session_summary(session_results)
            again = Prompt.ask("  Continue predicting?", choices=["y", "n"], default="y")
            if again == "n":
                break
        elif choice == "q":
            break
        # "p" → loop back to new journey

    # ── Goodbye ───────────────────────────────────────────────────────────────
    display_session_summary(session_results)
    console.print("\n[header]  Goodbye! 682  Safe travels.  [/header]\n")


if __name__ == "__main__":
    main()

Training Price model (1/3)…          ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:30
  Training Journey Status model (2/3)… ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:00:23
⠧ Training Delay Reason model (3/3)…   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0:04:06

✓ Models cached → /content/railway_models_v4.pkl

Model performance on held-out test data:

  Model                  Metric          Value     Rating         
 ───────────────────────────────────────────────────────────────── 
   Price                  MAE             £0.44     Excellent      
   Price                  RMSE            £1.65     Excellent      
   Price                  R²              0.997     Excellent      
   Journey Status         Acc            83.2%%     Good           
   Journey Status         F1              0.845     Good           
   Delay Reason           Acc            77.2%%     Excellent      
   Delay Reason           F1              0.774     Excellent

──────────────────────────────────────────────────   New Journey ──────────────────────────────────────────────────

...

╭─────────────────────────────────────────────── AI Recommendation ───────────────────────────────────────────────╮
│ High risk or poor value.                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────┬────────────────────────────────┬───────────────────────────────────────────────╮
│ Prediction                     │ Result                         │ Confidence                                    │
├────────────────────────────────┼────────────────────────────────┼───────────────────────────────────────────────┤
│ 💷  Estimated Price            │ £27.56                         │ Advance · Standard                            │
│ 🚦  Journey Status             │ ⚠️  Disrupted                   │ ███████████████████████░ 97.7%                │
│ 🔍  Delay Reason               │ ['Staff Shortage']             │ ████████████████████████ 98.7%                │
╰────────────────────────────────┴────────────────────────────────┴───────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│  Your £27.56 advance standard ticket carries a disruption risk: Disrupted predicted due to ['Staff Shortage']   │
│  (99%% confidence).                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

(p) Predict another  (s) Session summary  (q) Quit

What next? (p):